# Step 5a — Replication feasibility in GSE184950 (Wang et al. 2024)

**Project:** pd-da-aging · **Purpose:** count dopamine neurons per donor in GSE184950 before running the H1 replication. GSE157783 (the first-choice replication set) has only ~75 dopamine neurons in total, so it cannot be used; the switch is logged in docs/ANALYSIS_PLAN.md.

Run the cells **one at a time, top to bottom**, and stop at each 🛑. Results go to *My Drive › pd-da-aging_step5*. Runtime: CPU.

## Cell 1 — Connect Google Drive and install tools
Click **Allow** when Google asks. The red pip warning about pandas is harmless.

In [ ]:
# Connect Google Drive and install tools (1-2 minutes)
from google.colab import drive
drive.mount("/content/drive")
!pip install -q scanpy anndata


## Cell 2 — What GSE184950 provides on GEO
Lists the files on GEO and reads the sample information. Nothing large is downloaded yet.
### 🛑 Send Claude a screenshot of the file list and the sample table (scroll so all rows show).

In [ ]:
# Look at what GSE184950 provides on GEO (seconds)
import os, re, io, gzip, tarfile, json, datetime, sys, platform, urllib.request
import numpy as np, pandas as pd, scipy, anndata as ad
from scipy import sparse, io as spio

DRIVE = "/content/drive/MyDrive"
OUT = f"{DRIVE}/pd-da-aging_step5"
for sub in ["", "/da_candidates", "/tables", "/figures"]:
    os.makedirs(OUT + sub, exist_ok=True)
TODAY = datetime.date.today().isoformat()
with open(f"{OUT}/software_versions.txt", "w") as fh:
    fh.write(f"date: {TODAY}\npython: {sys.version.split()[0]}\nplatform: {platform.platform()}\n"
             f"numpy: {np.__version__}\npandas: {pd.__version__}\nscipy: {scipy.__version__}\nanndata: {ad.__version__}\n")

GEO = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE184nnn/GSE184950"
def fetch(url, stream=False):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    r = urllib.request.urlopen(req, timeout=300)
    return r if stream else r.read()

listing = fetch(f"{GEO}/suppl/").decode(errors="replace")
SUPPL = sorted(set(re.findall(r'href="(GSE184950[^"/]+)"', listing)))
print("Supplementary files on GEO:")
for f in SUPPL:
    m = re.search(re.escape(f) + r'</a>\s+\S+\s+\S+\s+(\S+)', listing)
    print(f"  {f}   {m.group(1) if m else ''}")

# Sample information from the GEO series matrix (same approach as step 2, Cell 4b)
mx = fetch(f"{GEO}/matrix/").decode(errors="replace")
rows = {}
for f in sorted(set(re.findall(r'href="(GSE184950[^"]*series_matrix\.txt\.gz)"', mx))):
    for line in gzip.decompress(fetch(f"{GEO}/matrix/{f}")).decode(errors="replace").splitlines():
        if line.startswith("!Sample_"):
            key, *vals = line.split("\t")
            rows.setdefault(key, []).extend(v.strip('"') for v in vals)
n = len(rows["!Sample_geo_accession"])
samples = pd.DataFrame({"gsm": rows["!Sample_geo_accession"], "title": rows["!Sample_title"]})
for key, vals in rows.items():
    if key.startswith("!Sample_characteristics") and len(vals) % n == 0:
        for k in range(len(vals) // n):
            for i, v in enumerate(vals[k * n:(k + 1) * n]):
                if ":" in v:
                    name, val = v.split(":", 1)
                    samples.loc[i, name.strip().lower()] = val.strip()
    if key in ["!Sample_source_name_ch1", "!Sample_description"] and len(vals) == n:
        samples[key.replace("!Sample_", "")] = vals
samples.to_csv(f"{OUT}/tables/GSE184950_samples.csv", index=False)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 40)
print(f"\n{len(samples)} samples. Columns: {list(samples.columns)}")
print(samples.to_string())


## Cell 3 — Count the dopamine neurons in every sample
Streams the GEO archive one sample at a time (no full copy is stored) and saves only the dopamine-neuron candidates to Drive. **This is the long one (about 20–60 minutes).** Keep the tab open; you can do other things meanwhile.
### 🛑 Send Claude a screenshot of the last lines when it says "Done".

In [ ]:
# Download each sample, count dopamine-neuron candidates, keep them (20-60 minutes; leave the tab open)
GENES = ["TH", "SLC6A3", "SLC18A2", "DDC", "SOX6", "CALB1", "AGTR1"]

def read_10x(files):
    """files: dict of short name -> bytes for barcodes, features/genes, matrix (any of .gz or plain)."""
    def text(b):
        return (gzip.decompress(b) if b[:2] == b"\x1f\x8b" else b).decode()
    mat = spio.mmread(io.BytesIO(gzip.decompress(files["matrix"]) if files["matrix"][:2] == b"\x1f\x8b" else files["matrix"])).tocsr()
    feats = pd.read_csv(io.StringIO(text(files["features"])), sep="\t", header=None)
    bcs = pd.read_csv(io.StringIO(text(files["barcodes"])), sep="\t", header=None)[0].astype(str).to_numpy()
    sym = feats[1] if feats.shape[1] > 1 else feats[0]
    if mat.shape == (len(sym), len(bcs)):
        mat = mat.T.tocsr()
    a = ad.AnnData(mat.astype(np.float32), obs=pd.DataFrame(index=bcs), var=pd.DataFrame(index=sym.astype(str).to_numpy()))
    a.var_names_make_unique()
    return a

def read_h5(b):
    import scanpy as sc, tempfile
    with tempfile.NamedTemporaryFile(suffix=".h5") as t:
        t.write(b); t.flush()
        a = sc.read_10x_h5(t.name)
    a.var_names_make_unique()
    return a

def role(name):
    n = name.lower()
    if n.endswith((".h5",)): return "h5"
    if "matrix.mtx" in n: return "matrix"
    if "barcodes" in n: return "barcodes"
    if "features" in n or "genes.tsv" in n: return "features"
    return None

def gsm_of(name):
    m = re.search(r"(GSM\d+)", name)
    return m.group(1) if m else os.path.basename(name).split("_")[0]

summary = []
def process(sample, a):
    a.obs_names = [f"{sample}_{b}" for b in a.obs_names]
    X = a.X.tocsr()
    tot = np.asarray(X.sum(1)).ravel(); ng = np.asarray((X > 0).sum(1)).ravel()
    g = {k: (np.asarray(X[:, a.var_names.get_loc(k)].todense()).ravel() if k in a.var_names else np.zeros(a.n_obs)) for k in GENES}
    keep_qc = ng >= 500
    da = keep_qc & (g["TH"] > 0) & ((g["SLC6A3"] > 0) | (g["SLC18A2"] > 0))
    row = {"gsm": sample, "nuclei": a.n_obs, "nuclei_500genes": int(keep_qc.sum()), "DA_candidates": int(da.sum()),
           "DA_AGTR1pos": int((da & (g["AGTR1"] > 0)).sum()), "DA_SOX6pos": int((da & (g["SOX6"] > 0)).sum()),
           "DA_CALB1pos": int((da & (g["CALB1"] > 0)).sum()),
           "DA_CALB1pos_SOX6neg": int((da & (g["CALB1"] > 0) & (g["SOX6"] == 0)).sum())}
    summary.append(row)
    if da.sum() > 0:
        sub = a[da].copy()
        sub.obs["total_counts_raw"] = tot[da]; sub.obs["n_genes_raw"] = ng[da]; sub.obs["gsm"] = sample
        sub.write_h5ad(f"{OUT}/da_candidates/{sample}_DA.h5ad", compression="gzip")
    print(f"  {sample}: {a.n_obs:,} nuclei, {row['DA_candidates']} DA candidates "
          f"(AGTR1+ {row['DA_AGTR1pos']}, CALB1+ {row['DA_CALB1pos']})", flush=True)

def walk_tar(tf, pending, outer_name=""):
    """Process a streamed tar: per-sample nested tar(.gz) archives, .h5 files, or loose 10x files."""
    for m in tf:
        if not m.isfile():
            continue
        name = m.name
        f = tf.extractfile(m)
        if name.endswith((".tar.gz", ".tgz", ".tar")):
            inner = tarfile.open(fileobj=f, mode="r|gz" if name.endswith(("gz", "tgz")) else "r|")
            files = {}
            for im in inner:
                r = role(im.name) if im.isfile() else None
                if r:
                    files[r] = inner.extractfile(im).read()
            s = gsm_of(name)
            if "h5" in files: process(s, read_h5(files["h5"]))
            elif {"matrix", "barcodes", "features"} <= set(files): process(s, read_10x(files))
            else: print(f"  {name}: no 10x matrix found inside ({sorted(files)})")
        else:
            r = role(name)
            if r is None:
                print(f"  skipped {name}"); continue
            s = gsm_of(name)
            if r == "h5":
                process(s, read_h5(f.read())); continue
            pending.setdefault(s, {})[r] = f.read()
            if {"matrix", "barcodes", "features"} <= set(pending[s]):
                process(s, read_10x(pending.pop(s)))

raw = [f for f in SUPPL if f.endswith("_RAW.tar")]
if not raw:
    print("No _RAW.tar on GEO for this series; send Claude a screenshot of Cell 2.")
else:
    print(f"Streaming {raw[0]} (no full copy is stored)...")
    pending = {}
    walk_tar(tarfile.open(fileobj=fetch(f"{GEO}/suppl/{raw[0]}", stream=True), mode="r|"), pending)
    if pending:
        print(f"Incomplete sample files left: {list(pending)}")
pd.DataFrame(summary).to_csv(f"{OUT}/tables/GSE184950_DA_counts_per_sample.csv", index=False)
print(f"\nDone: {len(summary)} samples processed.")


## Cell 4 — Is the replication feasible?
Dopamine-neuron candidates per sample and per diagnosis.
### 🛑 Send Claude a screenshot of the whole output.

In [ ]:
# Is the replication feasible? Dopamine neurons per donor and group (seconds)
cnt = pd.read_csv(f"{OUT}/tables/GSE184950_DA_counts_per_sample.csv")
s = pd.read_csv(f"{OUT}/tables/GSE184950_samples.csv")
t = s.merge(cnt, on="gsm", how="right")
print(t.drop(columns=[c for c in ["description"] if c in t.columns]).to_string())
diag_cols = [c for c in s.columns if re.search(r"diagnos|disease|status|condition|group|braak", c, re.I)]
print(f"\nColumns that look like diagnosis: {diag_cols}")
if diag_cols:
    dc = diag_cols[0]
    print(f"\nTotals by {dc}:")
    print(t.groupby(dc)[["nuclei", "nuclei_500genes", "DA_candidates", "DA_AGTR1pos", "DA_CALB1pos", "DA_CALB1pos_SOX6neg"]]
          .agg(["sum", "median"]).to_string())
print(f"\nAll samples: {int(t['DA_candidates'].sum()):,} DA candidates among {int(t['nuclei_500genes'].sum()):,} nuclei with >= 500 genes")
